In [ ]:
#This is pyenv file for making AI classification.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()
import time

In [2]:
data=pd.read_csv('data2015.csv')

In [4]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [5]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
3809,131264,17V540000,E-ONE,CYCLONE II,2015,NaN,STEERING:LINKAGES:TIE ROD ASSEMBLY,E-One Incorporated,20140822.0,20141114.0,...,E-One Incorporated (E-One) is recalling certai...,"If the tie rod loosens, it may disconnect from...","E-One will notify owners, and dealers will ins...",Owners may also contact the National Highway T...,74786001994138000000020,NaN,NaN,NaN,No,No
4456,141053,19V787000,MERCEDES-BENZ,CLS400,2015,2019120008,COMMUNICATION,"Mercedes-Benz USA, LLC.",20120120.0,20161006.0,...,"Mercedes-Benz USA, LLC. (MBUSA) is recalling c...",An inaccurate vehicle location may delay emerg...,MBUSA has initiated an Over the Air (OTA) remo...,Owners may also contact the National Highway T...,100407001984748000001419,NaN,NaN,NaN,No,No
3052,123277,17V262000,WINNEBAGO,VIEW,2015,NaN,ELECTRICAL SYSTEM,"Winnebago Industries, Inc",20140528.0,20160329.0,...,"Winnebago Industries, Inc (Winnebago) is recal...",The Axxess radio volume control module can aff...,"Winnebago will notify owners, and dealers will...",Owners may also contact the National Highway T...,68906001986690000000200,NaN,NaN,NaN,No,No
4675,145036,19V712000,THOMAS BUILT BUSES,SAF-T-LINER HDX,2015,FL-832,SEATS,Daimler Trucks North America LLC,20130705.0,20190815.0,...,Daimler Trucks North America LLC (DTNA) is re...,The lack of impact absorption increases the ri...,"DTNA will notify owners, and dealers will inst...",Owners may also contact the National Highway T...,99893001968095000000318,NaN,NaN,NaN,No,No
3913,132433,18V329000,VOLKSWAGEN,CC,2015,01C5/01C6,UNKNOWN OR OTHER,"Volkswagen Group of America, Inc.",20120119.0,20151118.0,...,"Volkswagen Group of America, Inc. (Volkswagen)...",If the vehicles do not meet all regulatory req...,"Volkswagen will notify owners, and dealers wil...",Owners may also contact the National Highway T...,86146002250719000000365,NaN,NaN,NaN,No,No


In [6]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

In [7]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [9]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [10]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-mini'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-mini):  


In [11]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [12]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $1.13'

In [13]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

def classify_fix(prompt_including_tweet):
    for attempt in range(3):
        try:
            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result
        except Exception as e:
            time.sleep(attempt*3)
    return None    

In [14]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify_fix)

100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [01:08<00:00,  1.37s/it]


In [15]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
3809,"E-One Incorporated (E-One) is recalling certain 2015 Cyclone II and Quest II emergency vehicles, equipped with certain Spicer D-Series and E-Series steer axles. The castellated nut on the steer axles may not be properly torqued, allowing the tie rod to loosen.",chasies
4456,"Mercedes-Benz USA, LLC. (MBUSA) is recalling certain 2013-2017 vehicles due to a software fault that may cause the date and time for the emergency call system (eCall) to differ from the actual date and time, potentially relaying an inaccurate vehicle location. For a full list of the affected models visit: https://static.nhtsa.gov/odi/rcl/2019/RMISC-19V787-4620.pdf",software
3052,"Winnebago Industries, Inc (Winnebago) is recalling certain 2014-2017 Winnebago View, VIA, ERA, and Itasca Navion motorhomes. The affected vehicles have an Axxess radio volume control module that may interfere with the chassis electrical system.",electronics
4675,"Daimler Trucks North America LLC (DTNA) is recalling certain 2014-2020 Thomas Built Saf-T-Liner EFX, Minotour, Saf-T-Liner C2, and Saf-T-Liner HDX school buses equipped with SynTec S3B or S3C seats. The seats may have been manufactured with styrene blocks that may not provide sufficient impact absorption in certain specific areas around the steel seat frame of the back support. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMVSS) nu...",safety_equipment
3913,"Volkswagen Group of America, Inc. (Volkswagen) is recalling certain 2012-2016 Eos, 2012 Passat, 2012-2016 CC, 2015-2016 e-Golf, 2011-2015 Touareg, 2012-2015 and 2017 Tiguan, and 2011-2016 Golf and 2011-2013 GTI vehicles. Modifications made while the vehicles were in an internal evaluation period may cause the affected vehicles to not comply with all of the applicable regulatory requirements.",others
6121,"Ford Motor Company (Ford) is recalling certain 2015-2017 Mustang vehicles. The rearview camera wiring may be loose or damaged, which can result in a blank or distorted image.",electronics
6253,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2013-2018 RAV4 vehicles. Certain replacement 12-volt batteries may not fit properly in the battery tray, allowing the battery to move and contact the hold-down bracket, possibly causing a short circuit.",electronics
6208,"General Motors, LLC (GM) is recalling certain 2003-2023 Chevrolet Express Cutaway and GMC Savana Cutaway incomplete vehicles equipped with wiring for an optional rear heating, ventilation, and air-conditioning (HVAC) system. When the HVAC blower motor circuit is left unused with exposed connectors and wiring, the circuit can corrode and overheat.",electronics
7096,"New Flyer of America, Inc. (New Flyer) is recalling certain XD35, XDE35, XE35, XN35, XD40, XDE40, XE40, XHE40, XN40, XT40, XD60, XDE60, XE60, XN60, XT60, and XHE60 transit buses. Please refer to New Flyer's recall report for specific model year information. The antilock braking (ABS) and electronic stability control (ESC) module may lose communication and fail without illuminating the warning lights. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety...",electronics
529,"BMW of North America, LLC (BMW) is recalling certain model year 2014-2015 MINI Cooper Hardtop 2 Door, and Cooper S Hardtop 2 Door vehicles manufactured January 6, 2014, to October 17, 2014. During service appointments, dealers may have inadvertently reprogrammed the transmission control unit with software that may allow drivers to exit the vehicle when the transmission is not in Park. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMV...",software


In [16]:
okay_to_do_the_whole_thing = input()

In [17]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [18]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify_fix)
data.to_csv(data_dir + 'Recall_classified2015.csv')

100%|██████████████████████████████████████████████████████████████████████████████| 7391/7391 [2:41:55<00:00,  1.31s/it]


In [3]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-fa3...4da",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 13.03190975,
    "include_byok_in_limit": false,
    "usage": 34.067152,
    "usage_daily": 6.96809025,
    "usage_weekly": 28.40863055,
    "usage_monthly": 32.1401981,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}
